# mPES - Colab Pro+ Bayesian-Optimisation Launcher

**Five cells. Set parameters -> run all -> close the browser.**

Workflow:
1. Edit `PKG`, `N_TRIALS`, optional `RESUME_DATE` and `GIT_REPO` in cell 1.
2. Run all cells (Runtime -> Run all).
3. Click **Runtime -> Manage sessions -> Background execution** (Pro+ only).
4. Close the browser. Re-run the **monitor** cell anytime to see live progress.

Open this notebook in **four Colab tabs** to run four packages in parallel.
Each instance writes its own SQLite DB to Drive, so they never collide.

The monitor cell at the bottom works **identically for every package** -
it auto-discovers the active run from `PKG` set in cell 1 and reads the
shared on-Drive artifacts (`run_meta.json`, `bayesian_opt.log`,
`optuna_study_<date>.db`).

In [ ]:
"""mPES Colab launcher notebook (cell 1: parameters and environment)."""
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
# ============================================================================
# Cell 1 - PARAMETERS (the only cell you normally edit)
# ============================================================================
PKG          = 'ql'                          # ql | dql | dqn | rdqn | ac | tr
N_TRIALS     = 100                           # target number of Optuna trials
RESUME_DATE  = ''                            # YYYY-MM-DD to resume; '' for fresh
GIT_REPO     = 'https://github.com/Maximiliano0/mPES_2026.git'
GIT_BRANCH   = 'models_improvement'
USE_GPU      = 0                             # 0=CPU (reproducible), 1=Colab GPU runtime

import os
os.environ['PKG']          = PKG
os.environ['N_TRIALS']     = str(N_TRIALS)
os.environ['RESUME_DATE']  = RESUME_DATE
os.environ['GIT_REPO']     = GIT_REPO
os.environ['GIT_BRANCH']   = GIT_BRANCH
os.environ['MPES_USE_GPU'] = str(USE_GPU)

# Sanity check: warn if USE_GPU=1 but no GPU is attached.
if USE_GPU == 1:
    import subprocess  # pylint: disable=import-outside-toplevel
    try:
        has_gpu = subprocess.run(
            ['nvidia-smi'], capture_output=True, check=False
        ).returncode == 0
    except FileNotFoundError:
        has_gpu = False
    if not has_gpu:
        print('[WARN] USE_GPU=1 but no GPU detected. '
              'Use Runtime -> Change runtime type -> GPU, then run all cells again.')
    else:
        print('[OK] GPU runtime detected - TF will use it.')
else:
    print('[INFO] Running on CPU (recommended for Colab -> local PC reproducibility).')
print(f'[CELL 1] env exported: PKG={os.environ["PKG"]!r} N_TRIALS={os.environ["N_TRIALS"]!r} RESUME_DATE={os.environ["RESUME_DATE"]!r}')


In [ ]:
# ============================================================================
# Cell 2 - MOUNT GOOGLE DRIVE
# ============================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)


In [ ]:
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
import subprocess  # noqa: E402
import os  # noqa: E402

# ============================================================================
# Cell 3 - CLONE/UPDATE REPO + INSTALL DEPS
# ============================================================================
_script = """
set -euo pipefail
REPO_DIR=/content/Win_mPES
if [[ ! -d "$REPO_DIR/.git" ]]; then
    git clone --branch "$GIT_BRANCH" "$GIT_REPO" "$REPO_DIR"
else
    cd "$REPO_DIR" && git fetch --all && git checkout "$GIT_BRANCH" && git pull
fi
cd "$REPO_DIR"
bash utils/colab/setup_colab.sh
"""

subprocess.run(_script, shell=True, executable='/bin/bash', check=True, env=os.environ.copy())


In [ ]:
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
import subprocess  # noqa: E402
import os  # noqa: E402

# ============================================================================
# Cell 4 - LAUNCH (detached supervisor + foreground monitor)
# ============================================================================
_script = '''
set -uo pipefail
cd /content/Win_mPES
bash utils/colab/run_colab.sh "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''

# check=False so we always see the script output instead of a generic CalledProcessError.
_proc = subprocess.run(
    _script, shell=True, executable='/bin/bash',
    check=False, env=os.environ.copy(),
)
if _proc.returncode != 0:
    print(f'\n[ERROR] run_colab.sh exited with code {_proc.returncode}.')
    print('        The supervisor may still be running detached on Drive.')
    print('        Inspect:  /content/drive/MyDrive/mPES/<PKG>/<DATE>_BAYESIAN_OPT/')
    print('                  {bayesian_opt.log, bayesian_opt_err.log, supervisor.log}')
